# Initialize Notebook Environment

Install NodeField's notebook dependencies into the selected kernel's Python environment.
Run the installation cell once per environment, or again after dependencies change.
Start Jupyter in the repository or one of its subdirectories.

The `full` extra includes training, notebook, synthetic, agent, ecosystem, and
Python-version-appropriate chemistry dependencies. Available sibling AbstractGraph
checkouts are installed editable; missing checkouts are resolved by pip through
NodeField's extras. Set `use_local_ecosystem = False` to use installed/index packages.

NSPPK is optional here: an installed package is retained, otherwise an available
sibling checkout is installed. Set `NODEFIELD_SKIP_NSPPK=1` to skip that installation.
Workflows that require NSPPK must have it installed before they run.

After installation, **restart the kernel**, then run only the verification/configuration
cell below. Other notebooks should use the same Python environment and run their own
`configure_notebook()` setup; Python variables and paths are not shared across kernels.


In [1]:
import importlib.util
import os
import subprocess
import sys
from pathlib import Path

print("First, I will find the NodeField project folder so its notebook dependencies can be installed.")
use_local_ecosystem = True
repo_root = next(
    (root for root in [Path.cwd(), *Path.cwd().parents]
     if (root / "conditional_node_field_graph_generator").is_dir()
     and (root / "pyproject.toml").is_file()),
    None,
)
if repo_root is None:
    raise RuntimeError("Start this notebook from the NodeField repository or a subdirectory.")

# Submit local checkouts and NodeField extras together so pip resolves their dependencies.
install_args = [sys.executable, "-m", "pip", "install", "-e", f"{repo_root}[full]"]
if use_local_ecosystem:
    print("Next, I will look for local AbstractGraph projects. Available copies will be linked so your edits are used by the notebooks.")
    ecosystem_root = repo_root.parent / "abstractgraph-ecosystem" / "repos"
    for name in ("abstractgraph", "abstractgraph-ml", "abstractgraph-graphicalizer"):
        checkout = ecosystem_root / name
        if (checkout / "pyproject.toml").is_file():
            print(f"Using the local {name} project at {checkout}")
            install_args.extend(["-e", str(checkout)])
        else:
            print(f"No local {name} project found. Pip will use an installed version or download a compatible release.")

print("Next, I will check whether NSPPK needs to be installed. Some graph workflows use this optional package.")
if os.environ.get("NODEFIELD_SKIP_NSPPK") != "1" and importlib.util.find_spec("nsppk") is None:
    nsppk_candidates = [
        parent / name
        for parent in (repo_root.parent, repo_root.parent / "INACTIVE")
        for name in ("NSPPK", "nsppk")
    ]
    local_nsppk_repo = next(
        (path for path in nsppk_candidates
         if (path / "pyproject.toml").is_file() or (path / "setup.py").is_file()),
        None,
    )
    if local_nsppk_repo is not None:
        print(f"NSPPK will be installed from {local_nsppk_repo}.")
        install_args.extend(["-e", str(local_nsppk_repo)])
    else:
        print("NSPPK was not found. Install it separately for workflows that require it.")

else:
    print("NSPPK is already available, or its installation was skipped with NODEFIELD_SKIP_NSPPK=1.")

print("I will now install NodeField and the packages used for training, plots, data loading, and graph workflows. This may take a few minutes.")
print(f"Installing notebook dependencies into {sys.executable}")
subprocess.check_call(install_args)
print("Installation complete. Restart the kernel so Python can load the newly installed packages, then run the verification cell below.")


First, I will find the NodeField project folder so its notebook dependencies can be installed.
Next, I will look for local AbstractGraph projects. Available copies will be linked so your edits are used by the notebooks.
Using the local abstractgraph project at /Users/f.costa/Code/abstractgraph-ecosystem/repos/abstractgraph
Using the local abstractgraph-ml project at /Users/f.costa/Code/abstractgraph-ecosystem/repos/abstractgraph-ml
Using the local abstractgraph-graphicalizer project at /Users/f.costa/Code/abstractgraph-ecosystem/repos/abstractgraph-graphicalizer
Next, I will check whether NSPPK needs to be installed. Some graph workflows use this optional package.
NSPPK is already available, or its installation was skipped with NODEFIELD_SKIP_NSPPK=1.
I will now install NodeField and the packages used for training, plots, data loading, and graph workflows. This may take a few minutes.
Installing notebook dependencies into /Users/f.costa/.venvs/py312/bin/python
Obtaining file:///Users/f

## Verify and configure after restarting the kernel

This cell is independent of the installation cell. Set `require_nsppk = True`
for NSPPK-based workflows and `require_chemistry = True` for molecular workflows.
On Python 3.13 and later, the project does not automatically request the chemistry
extra; molecular workflows still need a compatible RDKit installation.


In [2]:
import importlib
import sys
from pathlib import Path

require_nsppk = False
require_chemistry = False

repo_root = next(
    (root for root in [Path.cwd(), *Path.cwd().parents]
     if (root / "conditional_node_field_graph_generator").is_dir()
     and (root / "pyproject.toml").is_file()),
    None,
)
if repo_root is None:
    raise RuntimeError("Start this notebook from the NodeField repository or a subdirectory.")
if str(repo_root) not in sys.path:
    sys.path.insert(0, str(repo_root))

print("I will now check that the notebook packages can be loaded in this Python environment.")
modules = [
    "torch", "pytorch_lightning", "matplotlib", "requests", "yaml", "toolz", "openai",
    "abstractgraph", "abstractgraph_ml", "abstractgraph_graphicalizer",
]
if require_chemistry:
    modules.extend(["rdkit", "abstractgraph_graphicalizer.chem"])
for name in modules:
    print(f"Checking {name}...")
    try:
        importlib.import_module(name)
    except ImportError as exc:
        raise RuntimeError(
            f"Could not import {name} in {sys.executable}. "
            "Check the installation output and restart the kernel. "
            f"Original error: {exc}"
        ) from exc

from conditional_node_field_graph_generator.notebooks import configure_notebook

print("The package checks passed. Next, I will locate the data and output folders, create missing output folders, and report whether PyTorch can use CUDA.")
if require_nsppk:
    print("I will also check NSPPK because you enabled it for this workflow.")
context = configure_notebook(require_nsppk=require_nsppk)
print(f"Python: {sys.executable}")
for name, path in context.items():
    print(f"{name}: {path}")
print("Notebook environment verified.")


I will now check that the notebook packages can be loaded in this Python environment.
Checking torch...
Checking pytorch_lightning...
Checking matplotlib...
Checking requests...
Checking yaml...
Checking toolz...
Checking openai...
Checking abstractgraph...
Checking abstractgraph_ml...
Checking abstractgraph_graphicalizer...
The package checks passed. Next, I will locate the data and output folders, create missing output folders, and report whether PyTorch can use CUDA.
PyTorch version: 2.13.0
CUDA available: False
Python: /Users/f.costa/.venvs/py312/bin/python
REPO_ROOT: /Users/f.costa/Code/NodeField
ARTIFACT_ROOT: /Users/f.costa/Code/NodeField/.artifacts
NOTEBOOK_DATA_ROOT: /Users/f.costa/Code/NodeField/notebooks/datasets
CHECKPOINT_ROOT: /Users/f.costa/Code/NodeField/.artifacts/checkpoints/node_field
SAVED_GENERATOR_ROOT: /Users/f.costa/Code/NodeField/.artifacts/saved_generators
Notebook environment verified.
